# Day 6: タクシー配車アルゴリズムの比較

## 学習目標
- `TaxiHandler` の4種類の実装（`_random` / `_nearest` / `_nearest_matching_radious` /
  `_nearest_network_distance`）の違いを理解し、実際に比較する
- 配車の質を表す指標（マッチ率・平均待ち時間・平均車内時間）をどう読むかを身につける
- 台数（供給）とマッチング半径（サービス品質の制約）のトレードオフを実験する


In [ ]:
from uxsim import World
from uxsim.TaxiHandler import TaxiHandler_random, TaxiHandler_nearest, TaxiHandler_nearest_matching_radious
import random

def run_taxi_scenario(handler_cls, handler_kwargs=None, n_taxis=600, n_passengers=4000, imax=6, jmax=6, seed=0):
    handler_kwargs = handler_kwargs or {}
    deltan = 5
    tmax = 3600

    W = World(
        name="",
        deltan=deltan,
        tmax=tmax,
        print_mode=0, save_mode=0, show_mode=0,
        random_seed=seed,
    )
    nodes = {}
    for i in range(imax):
        for j in range(jmax):
            nodes[i, j] = W.addNode(f"n{(i,j)}", i, j, flow_capacity=1.6)
    for i in range(imax):
        for j in range(jmax):
            if i != imax - 1:
                W.addLink(f"l{(i,j,i+1,j)}", nodes[i, j], nodes[i+1, j], length=1000, free_flow_speed=20, number_of_lanes=1)
            if i != 0:
                W.addLink(f"l{(i,j,i-1,j)}", nodes[i, j], nodes[i-1, j], length=1000, free_flow_speed=20, number_of_lanes=1)
            if j != jmax - 1:
                W.addLink(f"l{(i,j,i,j+1)}", nodes[i, j], nodes[i, j+1], length=1000, free_flow_speed=20, number_of_lanes=1)
            if j != 0:
                W.addLink(f"l{(i,j,i,j-1)}", nodes[i, j], nodes[i, j-1], length=1000, free_flow_speed=20, number_of_lanes=1)

    rng = random.Random(seed)
    node_list = list(nodes.values())

    for _ in range(int(n_taxis / deltan)):
        node = rng.choice(node_list)
        W.addVehicle(node, None, 0, mode="taxi")

    handler = handler_cls(W, **handler_kwargs)
    for i in range(int(n_passengers / deltan)):
        o = rng.choice(node_list)
        d = rng.choice(node_list)
        while o == d:
            d = rng.choice(node_list)
        handler.add_trip_request(o, d, i / n_passengers * deltan * tmax / 2)

    while W.check_simulation_ongoing():
        W.exec_simulation(duration_t=60)
        handler.assign_trip_request_to_taxi()

    return W, handler

W_random, h_random = run_taxi_scenario(TaxiHandler_random)
print("=== TaxiHandler_random ===")
h_random.print_stats()


In [ ]:
W_nearest, h_nearest = run_taxi_scenario(TaxiHandler_nearest)
print("=== TaxiHandler_nearest ===")
h_nearest.print_stats()


In [ ]:
# マッチング半径つき: 半径外の空車は候補にしない(実務のマッチング半径制約に対応)
W_radius, h_radius = run_taxi_scenario(TaxiHandler_nearest_matching_radious, handler_kwargs={"matching_radious": 2.0})
print("=== TaxiHandler_nearest_matching_radious (radius=2.0) ===")
h_radius.print_stats()


## Part B: 演習

1. `matching_radious` を `1.0` / `2.0` / `5.0` で比較し、「マッチ率(`completed trip requests ratio`)」
   と「平均待ち時間」がどうトレードオフになるかをプロットしてください
   （半径を絞るほど、近くに空車がいない場合はマッチしにくくなるはずです）。
2. `n_taxis` を減らして供給不足の状況を作り、`TaxiHandler_nearest` の待ち時間がどう悪化するか
   確認してください。前回まとめた [EV充放電最適化 方針メモ] の
   「充電のためにタクシーが離脱する」状況は、これと似た供給不足問題として扱えます。
3. `trips_to_pandas()` を使ってリクエストごとの待ち時間分布をヒストグラムで可視化してください。


In [ ]:
import matplotlib.pyplot as plt

# TODO: matching_radiousを変えて「マッチ率 vs 平均待ち時間」のトレードオフをプロットする
# radii = [1.0, 2.0, 5.0]
# for r in radii:
#     W_r, h_r = run_taxi_scenario(TaxiHandler_nearest_matching_radious, handler_kwargs={"matching_radious": r})
#     h_r.print_stats()

df_trips = h_nearest.trips_to_pandas()
df_trips["waiting_time"].dropna().hist(bins=30)
plt.xlabel("waiting time [s]")
plt.ylabel("count")
plt.title("TaxiHandler_nearest: 待ち時間の分布")
plt.show()


## Part C: 考察

- `_nearest` は待ち時間の観点では良さそうですが、実務ではなぜ単純な最近傍配車だけでは
  不十分なことがあるのか考えてみてください（車両の偏在、公平性、将来需要の見込みなど）。
- Day 5 で学んだDUE/DSOの発想を配車問題に当てはめると、「個々の乗客に一番近いタクシーを
  割り当てる」ことと「ネットワーク全体の総待ち時間を最小化する」ことは、必ずしも
  同じ結果にならないはずです。どんな状況でズレが大きくなるか、具体例を考えてみてください。
